# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**Unit:** one row = one pseudonymized content item (page). Grain key is `content_id` (30,000 unique, 0 duplicates — see check). `client_id` groups rows (32 clients, unbalanced: 3–7,008 rows/client, median 567).

**Windows (trailing snapshot, export-anchored):**

- `*_90d` totals + `ctr / avg_position / engagement_rate / scroll_rate / ai_traffic_pct` — trailing 90 days.
- Label window: `impressions_last_30d` (days 1–30 back) vs `impressions_prev_30d` (days 31–60 back) → `trend_pct` → `trend_direction` → `is_declining_label`. The 90-day feature window therefore **contains** the label window: this snapshot supports observed association, not past→future prediction (warehouse work must use `*_prev30`-only features).
- Lifetime: `content_age_days` 90–564 (slice keeps only ≥90); `days_since_last_update` since last edit; `days_with_impressions` 1–88 of the 90 days.

In [7]:
from pathlib import Path
import pandas as pd

def _find_starter_csv():
    rel = Path("data/raw/content_refresh_anonymized.csv")
    cur = Path.cwd().resolve()
    for _ in range(8):
        cand = cur / rel
        if cand.exists():
            return cand
        if cur.parent == cur:
            break
        cur = cur.parent
    return None

RAW = _find_starter_csv()
assert RAW is not None, f"starter CSV not found from cwd={Path.cwd().resolve()} — run from the repo or set cwd to the repo root"
df = pd.read_csv(RAW)
print("shape:", df.shape, "| clients:", df["client_id"].nunique())
print("content_id unique:", df["content_id"].nunique(), "| dupes:", df.duplicated("content_id").sum())
print("rows/client describe:", df.groupby("client_id").size().describe().round(1).to_dict())
print("content_age_days min/max:", pd.to_numeric(df["content_age_days"], errors="coerce").min(), pd.to_numeric(df["content_age_days"], errors="coerce").max())
print("days_with_impressions min/max:", df["days_with_impressions"].min(), df["days_with_impressions"].max())
print("impressions_90d min:", pd.to_numeric(df["impressions_90d"], errors="coerce").min())


AssertionError: starter CSV not found from cwd=/content — run from the repo or set cwd to the repo root

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Label / proxy (never features):** `is_declining_label = (trend_direction == 'down')` — measured declining rate 54.2% (16,262/30,000). `trend_direction` (down/stable/up/new/flat = 16262/5962/4388/2236/1152) and `trend_pct` compute it; the six raw window inputs (`impressions/clicks/sessions_last_30d`, `impressions/clicks/sessions_prev_30d`) define the label window — never features.

**Features (27, exactly `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py`):** numeric: `search_volume, competition, cpc, word_count, char_count, log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct` (rates are x100: `ctr=0.76` means 0.76%; `avg_position=0` means no data, 1,205 rows). Categorical: `competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier`.

**Context (group / join / split / read-only):** `content_id, client_id` (pseudos — client-holdout splits, never features); raw totals read through their `log_` transforms (`impressions/clicks/sessions/ai_sessions_90d, pageviews/users/engaged_sessions/scroll_events_90d`); `age_tier_order, char_count_tier` (redundant tiers); prep flags `has_clicks, has_ai_sessions, measurable_opportunity`.

**Excluded (why):** `provider_used, model_used` — records which LLM generated the article (product-decision flag, 71.5%/19.1% blank), not page merit; learning it would rank provider routing instead of refresh need.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label_rate = (df["trend_direction"] == "down").mean()
print("declining (down) rate:", round(label_rate, 4), "|", (df["trend_direction"]=="down").sum(), "/", len(df))
print(df["trend_direction"].value_counts(dropna=False).to_dict())
# leakage identity: prev==0 <-> trend blank; new+flat == blank count
n_prev0 = (pd.to_numeric(df["impressions_prev_30d"], errors="coerce") == 0).sum()
n_blank = df["trend_pct"].isna().sum()
print("impressions_prev_30d==0:", n_prev0, "| trend_pct blank:", n_blank, "| new+flat:", ((df["trend_direction"]=="new")|(df["trend_direction"]=="flat")).sum())
# rates are x100 percentages
print("ctr median/mean/max:", pd.to_numeric(df["ctr"], errors="coerce").median(), round(pd.to_numeric(df["ctr"], errors="coerce").mean(),2), pd.to_numeric(df["ctr"], errors="coerce").max())


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

Claims and their checks (code below): grain holds (0 duplicate `content_id`); 30,000 rows / 32 clients; declining 54.2%; `avg_position=0` → 1,205 no-data rows; `trend_pct` blank 3,388 = `prev_30d==0` = new+flat — pattern, not random; missingness follows `content_type` (feedly 2,096 rows: 100% missing keyword fields; keyword articles: ~28.3% missing `word_count`, ~1.4% missing keyword data); `scroll_rate` blank 125 (pageviews=0); windows: age ≥90, impressions ≥1 (prep filter).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("GRAIN: dupes by content_id =", df.duplicated("content_id").sum(), "(0 => grain holds)")
print("COUNTS: rows =", len(df), "| clients =", df["client_id"].nunique())
print("MISSING overall %:", (df.isna().mean().sort_values(ascending=False).head(8)*100).round(1).to_dict())
print("MISSING word_count by type %:", (df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()*100)).round(1).to_dict())
print("MISSING search_volume by type %:", (df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()*100)).round(1).to_dict())
print("avg_position==0 (no data):", (df["avg_position"]==0).sum())
print("scroll_rate>100:", (pd.to_numeric(df["scroll_rate"], errors="coerce")>100).sum(), "| ai_traffic_pct>100:", (pd.to_numeric(df["ai_traffic_pct"], errors="coerce")>100).sum(), "(different measurement systems, not a bug)")
print("WINDOWS: age_tier =", df["age_tier"].value_counts(dropna=False).to_dict())


NameError: name 'df' is not defined

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Output:** this analysis hands the human a ranked refresh queue (`content_id`, score, reason codes) as decision-support — directional, not a claim about Google's algorithm.

**What this data can never tell you:** (1) unbalanced history — clients range 3–7,008 rows, so pooled averages reflect big clients; split by `client_id`. (2) No calendar dates ship with the starter slice — all windows are trailing/relative; use per-client windows on the warehouse (`gsc_data_start`, `IS TRUE` flag filters). (3) Label overlap — 90-day features contain the last-30-day label window; honest future-prediction needs `prev30`-only features. (4) Silent encodings — `avg_position=0` is missing (1,205), `trend_pct` blank 3,388 is structural (prev=0), rates >100 are valid (scroll 119, AI 23 rows); blind `fillna(0)` injects a `content_type` signal — use `has_*` flags.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("LIMIT client imbalance min/max:", df.groupby("client_id").size().min(), df.groupby("client_id").size().max())
print("LIMIT avg_position==0:", (df["avg_position"]==0).sum(), "| scroll blank:", df["scroll_rate"].isna().sum(), "| trend blank:", df["trend_pct"].isna().sum())
print("LIMIT scroll max:", pd.to_numeric(df["scroll_rate"], errors="coerce").max(), "| ai max:", pd.to_numeric(df["ai_traffic_pct"], errors="coerce").max())
print("OUTPUT: ranked refresh queue (content_id, score, reason codes) -> human review; decision-support only")


NameError: name 'df' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.